# rStar (Self-play muTuAl Reasoning)

## Components:
1. MCTS --> Smart gambler (could increase simulations per iteration to build confidence)
2. Beam Search --> Focuses the race, keeping the `k` most promising solution trajectories at each step
3. Rich Set of Reasoning Actions --> Promote human like decision making
4. Mutual Consistency -->  Peer2peer feedback (if you and your friend got the same answer. Then you have a higher confidence you are both right)
5. Tree Nodes --> Act as a logbook entry (stores all the vital information about a specific point in the problem solving process)
6. LLM --> Powerful AI consultant/expert
7. Smart judge --> Checks if 2 outputs are equivalent (0.5 vs 1/2 vs \frac{1}{2})

In [ ]:
from getpass import getpass
from dotenv import load_dotenv
import os
# Set up LLM connection

load_dotenv()
groq_key = os.getenv("GROQ_KEY", "Empty")

model = getpass("Enter the model name: ")
api_endpoint = getpass("Enter the API endpoint (default: https://api.openai.com): ")
#- Model should be "llama3-8b-8192"
#- endpoint should be "https://api.groq.com/openai"

api_endpoint = api_endpoint if api_endpoint else "https://api.openai.com"
api_key = groq_key

openai_api_base = f"{api_endpoint}/v1"

print(f"Model: {model}")
print(f"API Endpoint: {api_endpoint}")
print(f"OpenAI API Base: {openai_api_base}")
if api_key == "Empty":
    print("No API key needed.")
else:
    print(f"API Key Set")

Model: llama3-8b-8192
API Endpoint: https://api.groq.com/openai
OpenAI API Base: https://api.groq.com/openai/v1
API Key Set


In [2]:
from openai import OpenAI
import re # support for regex

# Initialize the OpenAI API client
client = OpenAI(
    api_key=api_key,
    base_url=openai_api_base
)

def chat_completion_request_openai(prompt): # function that will call the OpenAI API
    messages = [
        {"role": "user", "content": prompt}
    ]
    # Create chat completions using the OpenAI client
    chat_response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=1.0,
        max_tokens=1500,
    )

    # Extract the completion text from the response
    if chat_response.choices:
        completion_text = chat_response.choices[0].message.content
    else:
        completion_text = "No response from the model."
    return completion_text

# Test the endpoint
if __name__ == "__main__":
    # Test the function
    prompt = "Return an array of string, where each string is a season"
    chat_response = chat_completion_request_openai(prompt)
    print(f"Prompt: {prompt}\n")
    print(f"Response: {chat_response}")

Prompt: Return an array of string, where each string is a season

Response: Here is an array of strings, where each string represents a season:

`["Spring", "Summer", "Autumn", "Winter"]`

Let me know if you'd like me to reshape or manipulate this array in some way!


In [44]:
# Create prompt based on provided Action
def create_prompt_and_execute(question, state, action):
    prompts = {
            "A1": f"Question: {question}\n"
        f"Existing Reasoning Steps: {state}\n"
        "Please propose the next one-step thought to advance the reasoning process. "
        "Focus only on the immediate next step without solving the entire problem. "
        "Ensure the step is logically consistent with the existing reasoning steps. "
        "Do not provide the final answer or skip steps. Unless the next step clearly results in the final answer. "
        "Think step by step and provide only one reasoning step.",
            "A2": f"Question: {question}\n"
        f"Existing Reasoning Steps: {state}\n"
        "Please propose the remaining reasoning steps to solve the problem completely. "
        "Think step by step and ensure logical consistency with the existing reasoning steps. "
        "Provide the final answer at the end of the reasoning process. "
        "Do not skip steps or provide incomplete reasoning.",
            "A3": f"Question: {question}\n"
        f"Existing Reasoning Steps: {state}\n"
        "Please propose the next sub-question to simplify the problem further. "
        "After proposing the sub-question, provide its answer. "
        "Ensure the sub-question logically follows from the existing reasoning steps. "
        "Do not solve the entire problem or skip intermediate sub-questions. "
        "Think step by step and provide only one sub-question and its answer.",
        #     "A4": f"Sub-Question: {question}\n"
        # f"Original Answer: {state}\n"
        # "The original answer might be incorrect. "
        # "Please re-answer the sub-question using few-shot chain-of-thought reasoning. "
        # "Think step by step and ensure logical consistency in your reasoning. "
        # "Provide a detailed explanation and a verified final answer. "
        # "Do not reference the original answer in your response.",
            "A4": f"Sub-Question: {question}\\n"
        f"Existing Reasoning Steps: {state}\\n"
        "The sub-question might have been answered incorrectly. "
        "Please re-answer the last sub-question using few-shot chain-of-thought reasoning. "
        "Focus only on answering the sub-question accurately and logically. "
        "Do not reference the original answer or provide additional sub-questions. "
        "Provide a detailed explanation and a verified final answer to the sub-question. "
        "Do not provide the final answer to the original question or skip steps. Unless answering the sub-question clearly results in the final answer to the original question. "
        "Ensure logical consistency in your reasoning.",
            "A5": f"Original Question: {question}\n"
        "The original question might be misunderstood or unclear. "
        "Please rephrase the question to make it simpler and easier to understand. "
        "Clearly list all conditions and constraints provided in the problem statement. "
        "Ensure that no information is lost or altered during the rephrasing process. "
        "Do not solve the question or provide an answer.",
    }
    prompt = prompts[action] + "\n\nIf you determine the final answer, explicitly state 'The final answer is [your numeric answer]' at the end of your response."

    return chat_completion_request_openai(prompt) # return the new state


In [ ]:
import re

# Simulate a Node then rate the response (used to backpropagate the reward of an expanded node)
def simulate_then_rate(question, state, expected_answer):
    prompt = (
        f"Question: {question}\n"
        f"Existing Reasoning Steps: {state}\n"
        "Please propose the remaining reasoning steps to solve the problem completely. "
        "Think step by step and ensure logical consistency with the existing reasoning steps. "
        "Provide the final answer at the end of the reasoning process. "
        "Do not skip steps or provide incomplete reasoning."
    )
    simulated_answer = chat_completion_request_openai(prompt)
    print(f"Simulated Answer:\n{simulated_answer}\n")

    print(f"Total state:\n{state }\n\n{simulated_answer}\n")

    rating_prompt = (
        f"Question: {question}\n"
        f"Answer: {state}\n\n{simulated_answer}\n"
        f"Ground-truth Answer: {expected_answer}\n"
        "As an expert on this topic, please provide a detailed critique of the answer. "
        "Rate the answer based on correctness, completeness, and logical consistency. "
        "First state whether the answer is correct or incorrect. "
        "Provide only a critique, not a suggested answer. "
        "Then, rate the answer on a scale of 0 to 100. "
        "The response should be in the following format:\n"
        "Critique: <detailed critique>\n"
        "Rating: <rating>\n"
    )
    rating_response = chat_completion_request_openai(rating_prompt)
    print(f"Rating response:\n{rating_response}\n")

    # Extract the rating
    try:
        match = re.search(r"Rating:\s*(\d+)", rating_response) # Extract rating to be used in the UCT calculation
        if match:
            rating = int(match.group(1))
            if rating > 95: # Paper limits the rating to a maximum of 95 (maybe 96+ causes poor MCTS performance)
                rating = 95
            rating = float(rating)/100
        else:
            raise ValueError("Rating not found in the response.")
    except Exception as e:
        print(f"Error extracting rating: {e}")
        print(f"Rating response was: {rating_response}")
        rating = 0

    return rating # Simulated trajectory is not saved, only the rating is used to backpropagate the reward

# Test the function
if __name__ == "__main__":
    # Test the function
    question = "Solve the system of linear equations:\n\nx + 2y = 13\n3x - 4y = -18"
    ground_truth_answer = "x = 1.6, y = 5.7"
    # state = (
    #     "To solve the system of linear equations, we can use substitution or elimination methods.\n\n"
    #     "First, we will multiply the first equation by 3 and the second equation by 1 to make the coefficients of x in both equations equal:\n"
    #     "(3)(x + 2y) = (3)(13)\n"
    #     "(1)(3x - 4y) = (1)(-18)"
    # )
    state = (
        "To solve the system of linear equations, we can use substitution or elimination methods.\n\n"
        "First, we will multiply the first equation by 3 and the second equation by 1 to make the coefficients of x in both equations equal:\n"
        "(3)(x + 2y) = (3)(13)\n"
        "(1)(3x - 4y) = (1)(-18)\n\n"
        "Now, we will subtract the second equation from the first equation to eliminate the x variable:\n"
        "(3x + 6y) - (3x - 4y) = 39 - (-18)\n\n"
        "This simplifies to:\n"
        "10y = 57"
    )
    rating = simulate_then_rate(question, state, ground_truth_answer)
    print(f"Rating: {rating}")

    # Question URL --> https://huggingface.co/datasets/camel-ai/math/viewer/default/train?p=2&views%5B%5D=train&row=203
    # Ground-truth answer to the above question:
    # x = 1.6
    # y = 5.7

    # This is working as expected.

Simulated Answer:
Now that we have eliminated the x variable and have a simple equation with only y:

10y = 57

We can now solve for y by dividing both sides by 10:

y = 57/10
y = 5.7

Now that we have the value of y, we can substitute it back into one of the original equations to solve for x. Let's use the first equation:

x + 2y = 13

Substitute y = 5.7:

x + 2(5.7) = 13

Distribute 2 to 5.7:

x + 11.4 = 13

Subtract 11.4 from both sides to isolate x:

x = 13 - 11.4
x = 1.6

So, the solution to the system of linear equations is:

x = 1.6
y = 5.7

Therefore, the final answer is x = 1.6 and y = 5.7.

Total state:
To solve the system of linear equations, we can use substitution or elimination methods.

First, we will multiply the first equation by 3 and the second equation by 1 to make the coefficients of x in both equations equal:
(3)(x + 2y) = (3)(13)
(1)(3x - 4y) = (1)(-18)

Now, we will subtract the second equation from the first equation to eliminate the x variable:
(3x + 6y) - (3x

In [ ]:
import math
import random
import numpy as np

max_children = 4

class Node:
    def __init__(self, question, state, action=None, parent=None):
        self.state = state
        self.action = action # Action taken to reach this node
        self.parent = parent
        self.original_question = question
        self.current_question = question
        self.is_answered = False
        self.children = []
        self.visits = 0
        self.value = 0

    def is_fully_expanded(self):
        # Check if the node has reached the maximum number of children or if the answer has been found
        return len(self.children) >= max_children or self.is_answered or self.action == "A2"
    
    def best_child(self, exploration_weights=1.41):
        choices_weights = []
        for child in self.children:
            if child.visits == 0:
                weight = float('inf') # Prioritize unexplored nodes
            else:
                weight = (child.value / child.visits) + exploration_weights * math.sqrt(math.log(self.visits) / child.visits) # UCT calculation
                # Exploration term --> (child.value / child.visits) --> basically the average reward of the child
                # Exploitation term --> exploration_weights * math.sqrt(2 * math.log(self.visits) / child.visits) --> will be very high for unexplored nodes which will encourage exploration down that path
            choices_weights.append(weight)
        return self.children[np.argmax(choices_weights)]
    
    def most_visited_child(self): # This is used to pull the best trajectory from the MCTS tree once its generation is done (best trajectory == most likely to be the correct answer)
        return max(self.children, key=lambda child: child.visits) # Return the child with the most visits
    
    def add_child(self, child_node): # Utility function used to expand the tree
        self.children.append(child_node)

class rStar:
    def __init__(self, question, num_rollouts=1, max_depth=3, iterations=2):
        self.root = Node(question, "Begin answering the question")
        self.question = question
        self.num_rollouts = num_rollouts
        self.max_depth = max_depth
        self.iterations = iterations # Need to bound process by max_depth later

    def search(self):
        for i in range (self.iterations):
            print(f"Iteration {i+1}/{self.iterations}")
            node = self.select(self.root)
            print(f"Selected node: {node.state}")
            if not node.is_fully_expanded and not node.is_answered:
                node = self.expand(node)
                print(f"Expanded node: {node.state}")
            reward = self.simulate(node)
            print(f"Simulated reward: {reward}")
            self.backpropagate(node, reward)
        print(f"Visits to most visited child: {self.root.most_visited_child().visits}")
        return self.root.most_visited_child().state

    def select(self, node):
        while node.is_fully_expanded() and node.children:
            node = node.best_child() # Must check that the node is not answered in the best_child function
        return node
    
    def expand(self, node):
        actions = self.get_valid_actions(node)
        print(f"Valid actions: {actions}")

        for action in actions:
            action_prompt = create_prompt_and_execute(node.current_question, node.state, action)

        # Call create_prompt for each valid action
        # Expand every valid action

        # return a random child, which will be simulated
    
    def get_valid_actions(self, node):
        actions = ["A1", "A2", "A3", "A4", "A5"]

        if node.parent is None: # A4 cannot happen at the root
            actions.remove("A4")
        if node.parent is not None: # A5 can only can happen after the Root
            actions.remove("A5")
        if node.parent is not None and node.parent.action != "A3": # A4 can only happen after A3
            actions.remove("A4")

        for child in node.children: # Filter out actions that have already been taken --> shouldn't be taken
            if child.action in actions:
                actions.remove(child.action)
        
        return actions
        





In [49]:
# Testing the expand function

class Node:
    def __init__(self, question, state, action=None, parent=None):
        self.state = state
        self.action = action # Action taken to reach this node
        self.parent = parent
        self.original_question = question
        self.current_question = question
        self.is_answered = False
        self.children = []
        self.visits = 0
        self.value = 0

def expand(node):
        actions = get_valid_actions(node)
        print(f"Valid actions: {actions}")

        if node.action == "A2":
            print("A2 is a terminal action, no further expansion.")
            return # Terminal node
        
        # Call create_prompt for each valid action
        for action in actions:
            action_prompt = create_prompt_and_execute(node.current_question, node.state, action)
            print(f"{action} response:\n\n {action_prompt}")
            print("------------------------------------------")

        # Expand every valid action
        # return a random child, which will be simulated

def get_valid_actions(node):
        actions = ["A1", "A2", "A3", "A4", "A5"]

        if node.action == "A2": # A2 is a terminal action (since it full-shots)
             return []

        if node.parent is None: # A4 cannot happen at the root
            actions.remove("A4")
        if node.parent is not None: # A5 can only can happen after the Root
            actions.remove("A5")
        if node.parent is not None and node.parent.action != "A3": # A4 can only happen after A3
            actions.remove("A4")

        for child in node.children: # Filter out actions that have already been taken --> shouldn't be taken
            if child.action in actions:
                actions.remove(child.action)
        
        return actions

# Test the function
if __name__ == "__main__":
    question = "Solve the system of linear equations:\n\nx + 2y = 13\n3x - 4y = -18"

    root = Node("Solve the system of linear equations:\n\nx + 2y = 13\n3x - 4y = -18", "Begin answering the question", action=None, parent=None) # Expected actions ['A1', 'A2', 'A3', 'A5']
    node1 = Node(
        question,
        "The existing system of linear equations is:\n"
        "x + 2y = 13\n"
        "3x - 4y = -18\n"
        "My next step is to multiply the first equation by 4 and the second equation by 2, in order to eliminate the y-variable and make the coefficients of y additive in both equations.",
        action="A1",
        parent=root
    )  # Expected actions ['A1', 'A2', 'A3']
    node2 = Node( # Since A2 is supposed to one-shot the entire reasoning process, it should be the last node in the branch (no children allowed)
        question,
        "Let's solve the system of linear equations step by step.\n"
        "We are given the system of equations:\n"
        "x + 2y = 13 ... (Equation 1)\n"
        "3x - 4y = -18 ... (Equation 2)\n"
        "Our goal is to find the values of x and y.\n"
        "Step 1: Solve Equation 1 for x.\n"
        "x = 13 - 2y\n"
        "Step 2: Substitute the expression for x from Step 1 into Equation 2.\n"
        "3(13 - 2y) - 4y = -18\n"
        "Step 3: Expand and simplify the equation.\n"
        "39 - 6y - 4y = -18\n"
        "Step 4: Combine like terms.\n"
        "-10y = -57\n"
        "Step 5: Divide both sides by -10.\n"
        "y = 57/10\n"
        "y = 5.7\n"
        "Step 6: Substitute the value of y back into the expression for x from Step 1.\n"
        "x = 13 - 2(5.7)\n"
        "x = 13 - 11.4\n"
        "x = 1.6\n"
        "Step 7: The final answer is x = 1.6, y = 5.7.\n"
        "The final answer is x = 1.6, y = 5.7.",
        action="A2",
        parent=root
    )  # Expected actions []
    node3 = Node(
        question,
        "Let's begin by solving the system of linear equations.\n"
        "The given system of linear equations is:\n"
        "x + 2y = 13 ... (Equation 1)\n"
        "3x - 4y = -18 ... (Equation 2)\n"
        "To simplify the problem further, let's first try to eliminate one variable by making the coefficients of that variable the same in both equations.\n"
        "To eliminate the variable 'y', we can multiply Equation 1 by 2 and Equation 2 by 1, so that the coefficients of 'y' become the same:\n"
        "2(x + 2y = 13) => 2x + 4y = 26 ... (Multiplying Equation 1 by 2)\n"
        "3x - 4y = -18 ... (Equation 2) (without any changes)\n"
        "Sub-question: What is the result of subtracting Equation 2 from the modified Equation 1?\n"
        "Answer: Subtracting Equation 2 from the modified Equation 1, we get:\n"
        "2x + 4y = 26\n"
        "- (3x - 4y = -18)\n"
        "-----------------------------------\n"
        "6x = 44\n"
        "So, the result of subtracting Equation 2 from the modified Equation 1 is 6x = 44.\n"
        "Please let me know if you'd like me to proceed with the next step!",
        action="A3",
        parent=root
    )  # Expected actions ['A1', 'A2', 'A3', 'A4']
    node4 = Node(
        question,
        "Here is a rephrased version of the original question:\n"
        "Solve the system of two linear equations:\n"
        "1. x + 2y = 13\n"
        "2. 3x - 4y = -18\n"
        "The problem statement provides the following conditions and constraints:\n"
        "* There are two linear equations that need to be solved together.\n"
        "* The equations are:\n"
        "i. x + 2y = 13\n"
        "ii. 3x - 4y = -18\n"
        "* No additional information or constraints are provided.\n"
        "The rephrased question does not alter or lose any information from the original question.\n",
        action="A5",
        parent=root
    )  # Expected actions ['A1', 'A2', 'A3']
    node5 = Node( # Must come directly after A3
        question,
        "Some state",
        action="A4",
        parent=node3
    )  # Expected actions ['A1', 'A2', 'A3']

    # expand(root)
    expand(node1)
    # expand(node2) # Should not return anything since A3 is a terminal action
    # expand(node3)
    # expand(node4)
    # expand(node5)

    # I need to tweak the action_prompts to make sure that the output is properly structured for subsequent Nodes

    '''
    A3 response:

    I'm glad to help!
    Given the system of linear equations:
    x + 2y = 13
    3x - 4y = -18
    To simplify the problem further, I would like to propose the following sub-question:
    What is the value of x in terms of y?
    Answer: To find the value of x in terms of y, we can solve the first equation x + 2y = 13 for x:
    x = 13 - 2y
    This sub-question logically follows from the existing reasoning steps as it allows us to express x in terms of y, which can be useful in tackling the second equation.
    Next sub-question?
    '''
    # The last 1-2 lines should not be included in the response as they will hinder the state of following Node.

Valid actions: ['A1', 'A2', 'A3']
A1 response:

 The next step is to multiply the first equation by 4 and the second equation by 2, which would result in:

4x + 8y = 52
6x - 8y = -36

The next logical step would be to add the modified equations to eliminate the y-variable and make the coefficients of y additive in both equations.
------------------------------------------
A2 response:

 Based on the existing reasoning steps, I'll continue the solution:

1. Multiply the first equation by 4 to eliminate the y-variable:

4(x + 2y = 13)
4x + 8y = 52

2. Multiply the second equation by 2 to eliminate the y-variable:

2(3x - 4y = -18)
6x - 8y = -36

3. Now, add the modified equations to eliminate the y-variable:

4x + 8y = 52
6x - 8y = -36
----------------
10x = 16

4. Solve for x:

x = 16/10
x = 8/5

5. Now, substitute the value of x in one of the original equations to solve for y. I'll use the first equation:

x + 2y = 13
8/5 + 2y = 13

6. Solve for y:

2y = 13 - 8/5
2y = 51/5 - 8/5
2y = 4